# Лабораторная работа №2. Хост как сетевой узел

В этой работе мы не строим маршрутизаторы и не разбираем Ethernet-кадр по полям. Мы выясняем, **из каких частей складывается сетевое состояние обычного Linux-host**.

Главная модель: `link → address/prefix → connected route → neighbor → reachability`. Эти состояния связаны, но не заменяют друг друга. Проверять будем на реальных `pc1` и `pc2`.

## Схема стенда

![Схема Lab 002](assets/topology.svg)

`eth1` — учебный data plane. Адреса и MTU вы настроите сами. `eth0` нужен платформе для управления стендом — его не изменяем.

### Q1. Мы ещё не настраивали учебный IPv4 на `eth1`. Что вы ожидаете увидеть отдельно в `ip -br link show eth1` и `ip -br addr show eth1`? Если link окажется `UP`, что это докажет — и чего ещё не докажет?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q1, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

## 1. Сначала посмотрим, что уже существует

Откройте terminal `pc1` и выполните:

```bash
ip -br link show eth1
ip -br addr show eth1
ip link show eth1
ip -4 route show dev eth1
ip neigh show dev eth1
```

Ничего не настраивайте. Сейчас важен снимок состояния: интерфейс может существовать и быть `UP`, хотя учебного IPv4-адреса на нём ещё нет.

### Что мы разделяем

- `ip link` показывает состояние интерфейса и его L2-параметры;
- `ip addr` показывает L3-адреса и prefix;
- `ip route` показывает, куда kernel умеет направлять IP-трафик;
- `ip neigh` показывает известных L3-соседей на конкретном link.

В этой работе мы наблюдаем эти структуры как отдельные части состояния host. Механизм, которым Linux узнаёт MAC соседа, подробно разберём позже в лабораторной про ARP.

## 2. Добавим IPv4 и MTU

Сначала prediction. После команды `ip addr add 192.0.2.10/24 dev eth1` мы **не** будем вручную добавлять маршрут к `192.0.2.0/24`.

### Q2. Что вы ожидаете увидеть в `ip -4 route show dev eth1` сразу после назначения `192.0.2.10/24`? Почему эта строка может появиться без `ip route add`?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q2, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

### Настройте оба host

На `pc1`:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip link set eth1 mtu 1400
ip addr add 192.0.2.10/24 dev eth1
ip -br addr show eth1
ip link show eth1
ip -4 route show dev eth1
```

На `pc2`:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip link set eth1 mtu 1400
ip addr add 192.0.2.20/24 dev eth1
ip -br addr show eth1
ip link show eth1
ip -4 route show dev eth1
```

Сравните вывод с prediction. В частности найдите connected route к `192.0.2.0/24`.

### Наблюдение

После назначения `/24` kernel знает, что сеть `192.0.2.0/24` непосредственно подключена к `eth1`. Поэтому connected route является следствием состояния интерфейса и адреса, а не отдельной ручной настройки.

## Как накапливается состояние host

![Link, address, route и neighbor появляются не одновременно](assets/host-state.svg)

Анимация показывает последовательность, которую вы сейчас воспроизводите командами: link существует первым, адрес с prefix создаёт connected route, а neighbor state появляется после реального обмена.

## 3. Route уже есть. А neighbor пока может не быть

На `pc1` очистите только neighbor state и сразу посмотрите обе таблицы:

```bash
ip neigh flush dev eth1
ip -4 route show dev eth1
ip neigh show dev eth1
```


### Q3. Если neighbor table сейчас пуста, исчезнет ли connected route `192.0.2.0/24`? Что, по вашему прогнозу, изменится после первого успешного `ping 192.0.2.20`?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q3, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

### Создадим реальный трафик

Запустите следующую capture-cell. Пока она работает, в terminal `pc1` выполните:

```bash
ping -c 3 192.0.2.20
ip neigh show dev eth1
```

Capture здесь нужен не для разбора полей Ethernet — это тема следующей работы. Сейчас он подтверждает, что действие в terminal действительно породило пакеты на `eth1`.

In [ ]:
%%capture_traffic pc1:eth1 --filter "icmp" --timeout 10 --packets 30 --save captures/host-ping.pcap
import time
print("Сейчас выполните в terminal pc1: ping -c 3 192.0.2.20")
time.sleep(8)


### Наблюдение

Если `ping` прошёл, у нас есть сразу три разных факта: route выбрал локальный `eth1`, на интерфейсе реально прошёл ICMP-трафик, а `ip neigh` получил состояние соседа. Это связанные, но разные наблюдения.

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

viewer = PcapViewer("captures/host-ping.pcap", "icmp", limit=20)
frames = viewer.packets()
display(frames)
if frames.empty:
    print("ICMP не найден: повторите capture и ping во время захвата.")


### Route и neighbor отвечают на разные вопросы

`ip route` отвечает: **через какой интерфейс / next hop отправлять IP-трафик?**  
`ip neigh` отвечает: **какому L2-соседу на этом link соответствует нужный L3-адрес?**

Поэтому очистка neighbor table не удаляет connected route. И наоборот: наличие neighbor entry само по себе не заменяет корректный IPv4/prefix.

## 4. CHECK рабочего состояния

Перед CHECK быстро соберите доказательства на `pc1`:

```bash
ip -br link show eth1
ip -br addr show eth1
ip -4 route show dev eth1
ip neigh show dev eth1
ping -c 2 192.0.2.20
```

Теперь запустите **CHECK**. Он смотрит на наблюдаемое состояние обоих host: link, IPv4/prefix, MTU, connected route, reachability и neighbor state.

## 5. Controlled failure: link оставим живым, L3 сломаем

Сейчас удалим **только IPv4-адрес `pc2` на `eth1`**. Link и MTU специально не трогаем.

### Q4. После удаления `192.0.2.20/24` с `pc2:eth1` что должно остаться исправным, а что перестанет подтверждаться: `link UP`, MTU, IPv4 address, connected route на pc2, ping с pc1?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q4, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

На `pc2`:

```bash
ip -4 addr del 192.0.2.20/24 dev eth1
ip -br link show eth1
ip -br addr show eth1
ip -4 route show dev eth1
```

На `pc1`:

```bash
ping -c 2 -W 1 192.0.2.20
```

Запустите **CHECK** ещё раз. Не исправляйте сразу: посмотрите, какие проверки упали вместе и какая из них ближе всего к изменению, которое мы сделали.

### Диагностика по уровням

На `pc2` последовательно ответьте выводом команд:

```bash
ip -br link show eth1
ip link show eth1
ip -br addr show eth1
ip -4 route show dev eth1
```

Если `eth1` остаётся `UP` и MTU всё ещё `1400`, проблема не в существовании link. Исчезновение IPv4 и connected route локализует поломку на L3-состоянии host. Неуспешный ping — уже следствие этой причины.

## 6. Восстановление

На `pc2` верните только отсутствующее состояние:

```bash
ip addr add 192.0.2.20/24 dev eth1
ip -br addr show eth1
ip -4 route show dev eth1
```

На `pc1`:

```bash
ping -c 3 192.0.2.20
ip neigh show dev eth1
```

Запустите **CHECK** ещё раз. Финальное состояние должно снова пройти полностью.

### Q5. Почему одного успешного `ping` недостаточно, чтобы описать всё сетевое состояние host? Объясните, какую отдельную информацию дают link, address/prefix, route и neighbor table.

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q5, который находится выше.  
Не отвечай на исходный вопрос и не давай эталонный ответ.  
Проверь логику рассуждения, укажи на возможный пропущенный шаг и задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду следующего задания.

</details>

## Что нужно унести из Lab 002

- интерфейс может быть `UP`, но не иметь нужного IPv4;
- MAC, link state и MTU относятся к состоянию интерфейса, но не гарантируют L3-связность;
- IPv4 address + prefix создают connected route автоматически;
- route и neighbor table отвечают на разные вопросы;
- реальный traffic можно подтвердить capture на конкретном интерфейсе;
- одна первичная поломка может дать несколько симптомов в CHECK;
- диагностика начинается с проверки ближайшего изменённого состояния, а не со случайного набора команд.

В Lab 003 мы приблизимся к самому кадру Ethernet и разберём, что именно проходит по этому link.